---

# Extract.py

In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # notebooks/ 의 상위 = 프로젝트 루트 (src 를 찾기 위함)

from src.extract import extract_pdf_elements, categorize_elements, PROJECT_ROOT, fpath, fname

In [5]:
# 함수 실행 (hi_res 추출은 2~3분 소요)
raw_pdf_elements = extract_pdf_elements(fpath, fname)
texts, tables = categorize_elements(raw_pdf_elements)
print("전체 요소:", len(raw_pdf_elements), "/ 텍스트:", len(texts), "/ 표:", len(tables))

Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1
Ignoring wrong pointing object 2 65536 (offset 0)
Ignoring wrong pointing object 57 65536 (offset 0)
Ignoring wrong pointing object 88 65536 (offset 0)
Ignoring wrong pointing object 119 65536 (offset 0)
Ignoring wrong pointing object 135 65536 (offset 0)
Ignoring wrong pointing object 138 65536 (offset 0)
Ignoring wrong pointing object 141 65536 (offset 0)
Ignoring wrong pointing object 144 65536 (offset 0)
Ignoring wrong pointing object 147 65536 (offset 0)
Ignoring wrong pointing object 182 65536 (offset 0)
Ignoring wrong pointing object 198 65536 (offset 0)
Ignoring wrong pointing object 208 65536 (offset 0)
Ignoring wrong pointing object 244 65536 (offset 0)
Ignoring wrong pointing object 253 65536 (offset 0)
Ignoring wr

전체 요소: 13 / 텍스트: 7 / 표: 6


---

# Summarize.py 실행과정

In [6]:
import pickle
from dotenv import load_dotenv

from src.summarize import generate_text_summaries, generate_img_summaries

load_dotenv(PROJECT_ROOT / ".env")   # OPENAI_API_KEY 를 환경변수로 등록

True

In [7]:
# 텍스트·표 요약 (LLM 호출, 비용 발생)
with open(PROJECT_ROOT / "data" / "processed" / "extracted.pkl", "rb") as f:
    data = pickle.load(f)

text_summaries, table_summaries = generate_text_summaries(
    data["texts_2k_token"], data["tables"], summarize_texts=True
)
print("텍스트 요약:", len(text_summaries), "/ 표 요약:", len(table_summaries))

텍스트 요약: 27 / 표 요약: 35


In [8]:
# 이미지 요약 (VLM 호출, 비용 발생)
img_base64_list, image_summaries = generate_img_summaries(str(PROJECT_ROOT / "data" / "figures"))
print("이미지 요약:", len(image_summaries), "장")

이미지 요약: 10 장


# Retriever.py 실행과정

In [ ]:
from src.retriever import build_retriever, add_documents, build_chain, build_text_only_chain

In [ ]:
# 요약(서랍) + 원본(서가)을 같은 doc_id 로 연결해 색인 (임베딩 API 호출, 비용 발생)
# ⚠️ 이 셀은 실행할 때마다 컬렉션을 새로 비우고 다시 색인합니다 (중복 방지)
retriever = build_retriever()

if text_summaries:  add_documents(retriever, text_summaries,  data["texts_2k_token"])
if table_summaries: add_documents(retriever, table_summaries, data["tables"])
if image_summaries: add_documents(retriever, image_summaries, img_base64_list)
print("색인된 요약 수:", len(retriever.vectorstore.get()["ids"]))

In [ ]:
# 멀티모달 RAG 질의 (LLM 호출, 비용 발생)
chain = build_chain(retriever)
query = "그래프/표를 봐야만 답할 수 있는 질문을 적으세요"   # ← 실제 질문으로 바꿔서 실행
answer = chain.invoke(query)
print(f"Q: {query}
{'-'*70}
{answer}")

In [ ]:
# Before / After: 텍스트 전용 RAG 와 비교 (조건은 이미지·표 빼고 동일)
text_only_chain = build_text_only_chain(data["texts_2k_token"])
print("[BEFORE] 텍스트 전용 RAG
", text_only_chain.invoke(query))
print("[AFTER ] 멀티모달 RAG
",   chain.invoke(query))

---

---